# Transformer Zero: train, evaluate, benchmark (Colab T4)

Before **Runtime > Run all**: set **Runtime > Change runtime type > T4 GPU**, and add a secret named `HF_TOKEN` (key icon on the left, toggle *Notebook access* on). Every stage uploads its results to the private Hugging Face repo `wajih-rathore/transformer-zero`, so a disconnect loses at most one stage.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install -U transformers peft accelerate scikit-learn

In [ ]:
import os, subprocess
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
from huggingface_hub import snapshot_download, upload_folder
REPO = 'wajih-rathore/transformer-zero'
snapshot_download(REPO, allow_patterns=['code/*'], local_dir='/content/tz')
os.chdir('/content/tz/code')
def push(folder, path_in_repo):
    upload_folder(repo_id=REPO, folder_path=folder, path_in_repo=path_in_repo, commit_message=f'colab: {path_in_repo}')
    print('pushed', path_in_repo)
def run(cmd):
    print('$', cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)

## 1. Quick check on the small model (a few minutes)
Our decoder vs Hugging Face on Qwen3-0.6B, fp16, on this GPU.

In [ ]:
run('python scripts/benchmark.py --target Qwen/Qwen3-0.6B --draft Qwen/Qwen3-0.6B --parts correctness --repeats 1 --out results/smoke_0.6b.json')

## 2. Qwen3-4B: before fine-tuning, train, after fine-tuning

In [ ]:
M4 = 'Qwen/Qwen3-4B-Instruct-2507'
run(f'python scripts/evaluate.py --model {M4} --name qwen3-4b-base --splits val')
push('results', 'results')

In [ ]:
run(f'python scripts/train.py --model {M4} --out runs/qwen3-4b')
run('mkdir -p results/qwen3-4b-train && cp runs/qwen3-4b/train_log.json runs/qwen3-4b/val_metrics.json results/qwen3-4b-train/')
push('runs/qwen3-4b/best', 'qwen3-4b-lora')
push('results', 'results')

In [ ]:
run(f'python scripts/evaluate.py --model {M4} --adapter runs/qwen3-4b/best --name qwen3-4b-lora --splits val test')
run(f'python scripts/evaluate.py --model {M4} --name qwen3-4b-base --splits test')
push('results', 'results')

## 3. Inference benchmarks on the 4B (KV cache, attention, speculative decoding)

In [ ]:
run(f'python scripts/benchmark.py --target {M4} --draft Qwen/Qwen3-0.6B --adapter runs/qwen3-4b/best --out results/bench_qwen3-4b.json')
push('results', 'results')

## 4. Qwen3-1.7B: the second candidate, same recipe

In [ ]:
M17 = 'Qwen/Qwen3-1.7B'
run(f'python scripts/evaluate.py --model {M17} --name qwen3-1.7b-base --splits val test')
run(f'python scripts/train.py --model {M17} --out runs/qwen3-1.7b')
run('mkdir -p results/qwen3-1.7b-train && cp runs/qwen3-1.7b/train_log.json runs/qwen3-1.7b/val_metrics.json results/qwen3-1.7b-train/')
push('runs/qwen3-1.7b/best', 'qwen3-1.7b-lora')
run(f'python scripts/evaluate.py --model {M17} --adapter runs/qwen3-1.7b/best --name qwen3-1.7b-lora --splits val test')
push('results', 'results')